# Analizador de Forma en Sentadilla mediante Estimación de Pose y Reglas Biomecánicas

Clasificación de la técnica de ejecución de la sentadilla (*squat*) mediante **YOLOv8-pose** y un **clasificador basado en reglas biomecánicas ponderadas**.

## Resumen

Este notebook implementa un sistema de evaluación automática de la técnica de sentadilla mediante visión por computador. El sistema utiliza YOLOv8-pose para detectar y localizar 17 keypoints corporales por fotograma, a partir de los cuales se calculan ángulos articulares (rodilla, cadera, ángulo de espalda respecto a la vertical) y métricas de simetría bilateral. La clasificación de la ejecución se realiza mediante el `SquatFormClassifier`, un clasificador por reglas biomecánicas ponderadas que evalúa cinco criterios independientes y produce un puntaje global de 0 a 100. El clasificador no requiere datos de entrenamiento y genera retroalimentación interpretable por criterio. El modelo serializado se integra con la aplicación Streamlit desarrollada en `app.py`.

---

## Introducción

La evaluación de la técnica en ejercicios de fuerza como la sentadilla es fundamental para prevenir lesiones y maximizar el rendimiento. Los criterios de una ejecución correcta están bien documentados en la literatura biomecánica: profundidad de rodilla aproximada a 90°, espalda con inclinación moderada, rodillas alineadas sobre los pies y simetría entre ambos lados del cuerpo.

Automatizar esta evaluación mediante visión por computador presenta múltiples ventajas: accesibilidad (no requiere equipamiento especializado), escalabilidad (puede analizar múltiples personas simultáneamente) y objetividad (los criterios son aplicados de forma consistente).

Este notebook adopta un enfoque basado en **reglas biomecánicas interpretables**, en contraste con los enfoques de aprendizaje automático puro. Las razones de esta elección son:

1. **Aplicabilidad inmediata**: los umbrales se derivan de la literatura deportiva y no requieren un corpus de videos etiquetados para su funcionamiento.
2. **Transparencia**: cada decisión del clasificador puede explicarse en términos biomecánicos comprensibles para el usuario.
3. **Ajustabilidad**: los umbrales pueden calibrarse para distintos perfiles de ejecutante (nivel de habilidad, morfología corporal) sin reentrenar ningún modelo.

El clasificador implementado en `src/squat_classifier.py` evalúa cinco criterios ponderados: profundidad de rodilla (peso 2×), ángulo de espalda (peso 1×), alineación rodilla-tobillo (peso 1×), simetría bilateral (peso 1×) y estabilidad del movimiento (peso 0.5×). La comparación empírica con un clasificador XGBoost entrenado con videos etiquetados se documenta en `xgboost_training.ipynb`.

---
## 1. Importación de Módulos y Configuración

In [ ]:
# Imports 
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Nuestros módulos
sys.path.insert(0, '.')
from src.pose_extractor import PoseExtractor, KEYPOINT_NAMES
from src.angle_utils import get_squat_angles, aggregate_video_features, ANGLE_EXPLANATIONS
from src.squat_classifier import SquatFormClassifier, CriterionResult

# Config 
MODEL_NAME = "yolov8n-pose.pt"  
FRAME_SKIP = 3  # procesar 1 de cada N frames (más rápido)

print("✅ Imports listos")

---
## 2. Carga del Modelo YOLOv8-Pose

In [ ]:
# Cargar el modelo de pose
pose_extractor = PoseExtractor(MODEL_NAME)
print(f"Modelo cargado: {MODEL_NAME}")
print(f"Keypoints COCO: {len(KEYPOINT_NAMES)}")
for kid, name in KEYPOINT_NAMES.items():
    print(f"   {kid}: {name}")

---
## 3. Prueba de Extracción de Keypoints

Se verifica el funcionamiento del extractor de pose sobre un frame de muestra. Si no se dispone de cámara, se genera un frame de demostración sintético.

In [ ]:
# Función para capturar desde webcam
def capture_webcam(camera_id=0):
    cap = cv2.VideoCapture(camera_id)
    if not cap.isOpened():
        print("No se pudo abrir la cámara")
        return None
    print("📸 Capturando frame...", end=" ")
    ret, frame = cap.read()
    cap.release()
    if ret:
        print("OK!")
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    else:
        print("Error al capturar")
        return None

# Intentar con la cámara
frame = capture_webcam()

if frame is None:
    # Crear un frame sintético
    print("ℹ No hay cámara — creando frame de demostración")
    frame = np.ones((480, 640, 3), dtype=np.uint8) * 200
    cv2.putText(frame, "Sin camara - usa un video", (100, 240),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

print(f"   Frame shape: {frame.shape}")

In [ ]:
# Extraer keypoints
kps, annotated = pose_extractor.extract_from_frame(frame)

print(f"Keypoints detectados: {len(kps)}")
for kp in kps[:5]:
    name = KEYPOINT_NAMES.get(kp['id'], f"kp_{kp['id']}")
    print(f"   {kp['id']:2d} {name:20s} x={kp['x']:7.1f} y={kp['y']:7.1f} conf={kp['confidence']:.2f}")
if len(kps) > 5:
    print(f"   ... y {len(kps) - 5} más")

# Mostrar resultado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(frame)
axes[0].set_title('Frame original')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title('Pose estimada')
axes[1].axis('off')
plt.tight_layout()
plt.show()

---
## 4. Cálculo de Ángulos Biomecánicos

A partir de los keypoints detectados, se calculan las métricas biomecánicas específicas para la evaluación de la sentadilla:

- **Ángulo de rodilla**: ángulo formado en la articulación rodilla (cadera → rodilla → tobillo), para ambos lados.
- **Ángulo de cadera**: ángulo formado en la articulación de cadera (hombro → cadera → rodilla), para ambos lados.
- **Ángulo de espalda**: inclinación del torso respecto al eje vertical.
- **Desplazamiento rodilla-tobillo**: diferencia horizontal entre la posición de la rodilla y el tobillo (knee-over-toe).
- **Asimetría bilateral**: diferencia entre el lado izquierdo y el derecho para rodilla y cadera.

In [ ]:
# Convertir keypoints a dict y calcular ángulos
if kps:
    kps_dict = pose_extractor.keypoints_to_dict(kps)
    angles = get_squat_angles(kps_dict)
    
    print("Ángulos calculados:\n")
    for key, value in angles.items():
        print(f"   {key:25s} = {value:8.2f}")
        if key in ANGLE_EXPLANATIONS:
            print(f"   {'':25s}   {ANGLE_EXPLANATIONS[key][:80]}...")
        print()
else:
    print("No hay keypoints para calcular ángulos")
    print("(Es normal si no hay una persona visible en el frame)")

---
## 5. Procesamiento de Video

Si se dispone de un video de sentadilla, se procesa con el extractor de pose para obtener los keypoints por frame. Los videos deben ubicarse en el directorio `test_videos/`.

In [ ]:
# Buscar videos
video_dir = Path("test_videos")
videos = list(video_dir.glob("*.mp4")) + list(video_dir.glob("*.MP4")) + list(video_dir.glob("*.avi"))

if videos:
    print("📹 Videos encontrados:")
    for i, v in enumerate(videos):
        print(f"   [{i}] {v.name}")
else:
    print("No hay videos en 'test_videos/'")
    print("Copia un video .mp4 y vuelve a ejecutar esta celda.")
    print("O usa la sección 'Demo Sintética' más abajo.")

### Opción A: Procesar un Video Real

Disponible cuando se cuenta con archivos de video en el directorio `test_videos/`.

In [ ]:
# Procesar un video
# video_path = str(videos[0])
# print(f"Procesando: {video_path}")
# 
# frame_keypoints, output_video = pose_extractor.process_video(
#     video_path, frame_skip=FRAME_SKIP)
# print(f"Frames procesados: {len(frame_keypoints)}")
# print(f"Video anotado: {output_video}")

print("Descomentar el código de arriba cuando se tenga un video.")

### Opción B: Demostración con Datos Sintéticos

Se generan datos de ángulos sintéticos para demostrar el funcionamiento del clasificador sin necesidad de videos reales. Los datos sintéticos simulan la progresión temporal de una sentadilla completa (posición de pie → posición de fondo → posición de pie) con parámetros diferenciados para ejecución correcta y ejecución deficiente.

In [ ]:
# Generar datos sintéticos de ejemplo
np.random.seed(42)

def generate_synthetic_squat(is_good: bool, n_frames: int = 50):
    """Genera ángulos sintéticos para una sentadilla."""
    frames = []
    for i in range(n_frames):
        # Progresión: de pie (0%) → abajo (50%) → de pie (100%)
        phase = i / n_frames
        depth = np.sin(phase * np.pi)  # 0 → 1 → 0
        
        if is_good:
            knee_min = 80 + np.random.uniform(-5, 5)       # ~80° en el fondo
            back = 30 + depth * 10 + np.random.uniform(-3, 3)  # 30-40°
            sym = np.random.uniform(0, 5)                  # simétrica
            ktoe = np.random.uniform(-10, 20)              # rodilla controlada
        else:
            knee_min = 120 + np.random.uniform(-5, 5)      # no llega a paralelo
            back = 50 + depth * 20 + np.random.uniform(-5, 5)  # muy inclinado
            sym = np.random.uniform(10, 25)                # asimétrico
            ktoe = np.random.uniform(-30, 50)              # rodilla descontrolada
        
        # Ángulo de rodilla: de pie (~170°) → fondo (~knee_min) → de pie
        knee = 170 - (170 - knee_min) * depth + np.random.uniform(-3, 3)
        
        frames.append({
            'left_knee_angle': knee + np.random.uniform(-1, 1),
            'right_knee_angle': knee + (0 if is_good else sym),
            'left_hip_angle': 120 - depth * 40 + np.random.uniform(-2, 2),
            'right_hip_angle': 120 - depth * 40 + (0 if is_good else sym),
            'back_angle': back + np.random.uniform(-2, 2),
            'left_knee_toe_x': ktoe + np.random.uniform(-5, 5),
            'right_knee_toe_x': ktoe + np.random.uniform(-5, 5),
            'knee_symmetry': abs(sym + np.random.uniform(-1, 1)),
            'hip_symmetry': abs(sym * 0.8 + np.random.uniform(-1, 1)),
        })
    return frames

# Generar samples
good_frames = generate_synthetic_squat(is_good=True)
bad_frames = generate_synthetic_squat(is_good=False)

good_features = aggregate_video_features(good_frames)
bad_features = aggregate_video_features(bad_frames)

print(f" Datos sintéticos generados:")
print(f" Buena forma: {len(good_frames)} frames")
print(f" Mala forma: {len(bad_frames)} frames")
print(f"\n  Features totales por video: {len(good_features)}")

---
## 6. Clasificador de Forma

Se instancia el `SquatFormClassifier` y se evalúan las características biomecánicas calculadas. El clasificador devuelve un puntaje por criterio, un puntaje global ponderado (0–100) y retroalimentación específica por aspecto de la ejecución.

In [ ]:
# Crear clasificador
classifier = SquatFormClassifier()
print("Clasificador creado")
print(f"Umbrales: {len(classifier.thresholds)} parámetros")

In [ ]:
def show_evaluation(classifier, features, label: str):
    """Muestra la evaluación completa para un set de features."""
    criteria, overall = classifier.score_squat(features)
    pred = classifier.predict([features])[0]
    proba = classifier.predict_proba([features])[0]
    
    print(f"{'='*60}")
    print(f" EVALUACIÓN: {label}")
    print(f"{'='*60}")
    print(f"   Predicción: {' Buena forma' if pred == 0 else 'Mala forma'}")
    print(f"   Confianza:  {proba[0]:.1%} buena / {proba[1]:.1%} mala")
    print(f"   Puntaje global: {overall:.1f}/100\n")
    
    print(f"   {'Criterio':25s} {'Puntaje':8s} {'Peso':6s}  {'Resultado'}")
    print(f"   {'─'*25} {'─'*8} {'─'*6}  {'─'*40}")
    for c in criteria:
        status = 'Ok' if c.passed else 'Mejorar'
        print(f"   {c.name:25s} {c.score:6.1f}/100 {c.weight:4.1f}x  {status} {c.detail[:60]}")
    print()
    
    feedback = classifier.get_feedback(features)
    print("Feedback:")
    for tip in feedback:
        print(f"   {tip}")
    print()

# Evaluar ambos casos
show_evaluation(classifier, good_features, "BUENA FORMA (sintética)")
show_evaluation(classifier, bad_features, "MALA FORMA (sintética)")

---
## 7. Serialización del Modelo

El clasificador se guarda en formato joblib para su uso en la aplicación Streamlit (`app.py`).

In [ ]:
model_path = Path("models/squat_form_model.pkl")
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(classifier, model_path)
print(f" Modelo guardado: {model_path}")
print(f" Tamaño: {model_path.stat().st_size / 1024:.1f} KB")

---
## 8. Verificación de Carga

Se verifica que el modelo serializado se carga correctamente y produce predicciones consistentes con las obtenidas antes de la serialización.

In [ ]:
# Cargar modelo guardado
loaded = joblib.load(model_path)

# Probar que funciona
test_features = [good_features, bad_features]
preds = loaded.predict(test_features)
probas = loaded.predict_proba(test_features)

print(" Modelo cargado y verificado")
print(f"   Predicciones: {preds}")
print(f"   Probabilidades:\n")
for i, (p, pr) in enumerate(zip(preds, probas)):
    label = "Buena forma" if p == 0 else "Mala forma"
    print(f"   Sample {i}: {label} (good={pr[0]:.1%}, bad={pr[1]:.1%})")

---
## 9. Análisis en Tiempo Real

Esta sección contiene una función para procesar frames de video en vivo desde una cámara conectada. Su uso es opcional y requiere un entorno con interfaz gráfica disponible. Para activar el análisis en tiempo real, descomentar la llamada a `live_webcam_analysis()` al final de la celda. Presionar la tecla `q` para finalizar la sesión.

In [ ]:
def live_webcam_analysis(camera_id=0):
    """Análisis en vivo desde la webcam."""
    cap = cv2.VideoCapture(camera_id)
    if not cap.isOpened():
        print("No se pudo abrir la cámara")
        return
    
    print("📸 Análisis en vivo — Presioná 'q' para salir")
    frame_count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
        
        # Procesar cada frame (sin skip para tiempo real)
        kps, annotated = pose_extractor.extract_from_frame(frame)
        
        # Calcular ángulos si hay keypoints
        if kps:
            kps_dict = pose_extractor.keypoints_to_dict(kps)
            angles = get_squat_angles(kps_dict)
            
            # Mostrar ángulos en el frame
            y_pos = 30
            for key, val in angles.items():
                if 'knee' in key and 'angle' in key:
                    cv2.putText(annotated, f"{key}: {val:.0f}",
                                (10, y_pos), cv2.FONT_HERSHEY_SIMPLEX,
                                0.5, (0, 255, 0), 1)
                    y_pos += 20
        
        cv2.imshow("Squat Form Analyzer - Presiona 'q' para salir", annotated)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    print(f"Sesión finalizada — {frame_count} frames procesados")


# Ejecutar análisis en vivo (descomentar para usar)
# live_webcam_analysis()

print(" Descomentar la línea de arriba para activar la webcam.")
print(" Se necesita ejecutar en un entorno con display (no en terminal remota).")

---
## Conclusiones

### Resultados del notebook

| Componente | Estado |
|---|---|
| Extracción de keypoints con YOLOv8-pose | Verificado |
| Cálculo de ángulos biomecánicos por frame | Verificado |
| Clasificador basado en reglas biomecánicas | Funcional |
| Modelo serializado en `models/squat_form_model.pkl` | Generado |
| Compatibilidad con `app.py` | Confirmada |

### Análisis del clasificador de reglas

El `SquatFormClassifier` demuestra las ventajas del enfoque basado en reglas interpretables para este problema:

- **No requiere datos etiquetados**: los umbrales se derivan de la literatura biomecánica y son aplicables desde la primera ejecución.
- **Retroalimentación interpretable**: cada criterio produce un mensaje específico que indica al usuario qué aspecto de su técnica debe corregir (por ejemplo, "ángulo de rodilla: 125° — no se alcanzó la posición paralela").
- **Ajustabilidad**: los umbrales pueden modificarse en el diccionario `DEFAULT_THRESHOLDS` de `src/squat_classifier.py` sin necesidad de reentrenar ningún modelo.
- **Tamaño mínimo**: el modelo serializado ocupa menos de 1 KB, en contraste con modelos de aprendizaje automático que pueden requerir megabytes de almacenamiento.

### Limitaciones

- Los umbrales son fijos y no se adaptan automáticamente a variaciones en la morfología corporal o el nivel de habilidad del ejecutante.
- La calidad de la evaluación depende directamente de la calidad de la detección de pose: visibilidad de los keypoints, iluminación y ángulo de la cámara.
- El clasificador evalúa la sentadilla promediando los ángulos a lo largo del tiempo; no identifica errores que ocurren en fases específicas del movimiento (por ejemplo, solo durante el descenso).

### Relación con el enfoque de aprendizaje automático

Para la comparación empírica entre este clasificador de reglas y un modelo XGBoost entrenado con videos etiquetados, consultar el notebook `xgboost_training.ipynb`. Los resultados de esa comparación muestran que, con el volumen de datos disponible (19 videos), ambos enfoques alcanzan un rendimiento equivalente sobre el conjunto de prueba (Accuracy 66.7%), lo que valida la calidad de los criterios biomecánicos implementados en este notebook.